In [ ]:
# Cell 1: Imports and configuration
from dataclasses import dataclass
from pathlib import Path
import wave

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Data directories (folder names are kept exactly as they are on disk)
TRAIN_DIR = Path("../../TinHieuHuanLuyen")
TEST_DIR = Path("../../TinHieuKiemThu")
OUTPUT_DIR = Path("output/mine")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DSP parameters fixed by the instructor
FRAME_MS = 25         # frame length (ms)
HOP_MS = 10           # frame hop (ms)
MIN_SILENCE_MS = 200  # silences shorter than this are treated as speech

# Boundary padding in frames (1 frame = HOP_MS = 10 ms).
# Not specified by the paper (it pads 5 frames of a 50 ms window = 250 ms), so it is a free
# parameter: chosen by sweeping on TRAIN only (see Cell 7), never tuned on TEST.
PAD_START = 1
PAD_END = 1

In [ ]:
# Cell 2: Data loading and feature extraction (Giannakopoulos 2014)
@dataclass
class Sample:
    """One audio file with its precomputed features and ground-truth speech boundaries."""
    name: str
    signal: np.ndarray
    fs: int
    ste: np.ndarray            # normalized short-time energy, one value per frame
    sc: np.ndarray             # normalized spectral centroid, one value per frame
    frame_starts: np.ndarray   # start time of each frame (s)
    gt: tuple                  # ground-truth (start, end) in seconds

    @property
    def duration(self):
        return len(self.signal) / self.fs


def read_wav(wav_path):
    """Read a 16-bit PCM WAV file; return the signal scaled to [-1, 1] and the sampling rate."""
    with wave.open(str(wav_path), "rb") as wf:
        fs = wf.getframerate()
        raw = wf.readframes(wf.getnframes())
    signal = np.frombuffer(raw, dtype=np.int16).astype(float)
    return signal / max(np.max(np.abs(signal)), 1.0), fs


def read_speech_bounds(lab_path):
    """Return the ground-truth (start, end) in seconds spanning all voiced ('v') and unvoiced ('uv') segments."""
    segments = []
    for line in lab_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) == 3 and parts[2].lower() in {"v", "uv"}:
            segments.append((float(parts[0]), float(parts[1])))
    return round(min(s for s, _ in segments), 4), round(max(e for _, e in segments), 4)


def compute_features(signal, fs, frame_ms=FRAME_MS, hop_ms=HOP_MS):
    """
    Extract the two frame-level features of the paper, each normalized to [0, 1]:
      - Short-Time Energy:  E(i) = (1/N) * sum(|x_i(n)|^2)
      - Spectral Centroid:  C(i) = sum(k * X_i(k)) / sum(X_i(k)), on the DFT magnitude
    Returns (ste, sc, frame_starts); frame_starts is in seconds.
    """
    frame_size = int(round(frame_ms * fs / 1000.0))
    hop_size = int(round(hop_ms * fs / 1000.0))
    n_frames = max(1, int(np.ceil(max(0, len(signal) - frame_size) / hop_size)) + 1)

    bin_index = np.arange(1, frame_size // 2 + 2)  # 1-based DFT bin numbers (rfft has N//2 + 1 bins)
    n_bins = len(bin_index)

    ste = np.zeros(n_frames)
    sc = np.zeros(n_frames)
    frame_starts = np.arange(n_frames) * hop_size / fs

    for i in range(n_frames):
        chunk = signal[i * hop_size : i * hop_size + frame_size]
        frame = np.zeros(frame_size)
        frame[:len(chunk)] = chunk  # zero-pad the last frame

        ste[i] = np.mean(frame ** 2)

        magnitude = np.abs(np.fft.rfft(frame))
        total = magnitude.sum()
        if total > 1e-12:
            sc[i] = np.sum(bin_index * magnitude) / (total * n_bins)

    return ste / max(ste.max(), 1e-12), sc / max(sc.max(), 1e-12), frame_starts


def load_dataset(data_dir):
    """Load every .wav in data_dir together with its .lab ground truth, and precompute its features."""
    samples = []
    for wav_path in sorted(data_dir.glob("*.wav")):
        signal, fs = read_wav(wav_path)
        ste, sc, frame_starts = compute_features(signal, fs)
        gt = read_speech_bounds(wav_path.with_suffix(".lab"))
        samples.append(Sample(wav_path.name, signal, fs, ste, sc, frame_starts, gt))
    return samples


train_set = load_dataset(TRAIN_DIR)
test_set = load_dataset(TEST_DIR)
print(f"Loaded {len(train_set)} train files and {len(test_set)} test files")

In [ ]:
# Cell 3: Two-feature histogram VAD (Giannakopoulos 2014)
@dataclass
class HistogramThreshold:
    """Adaptive threshold of one feature, plus the histogram data it was derived from."""
    value: float          # T = (W * M1 + M2) / (W + 1)
    m1: float             # first local maximum of the smoothed histogram (silence peak)
    m2: float             # second local maximum (speech peak)
    centers: np.ndarray   # histogram bin centers
    smooth: np.ndarray    # smoothed histogram counts


@dataclass
class VADResult:
    start: float            # predicted speech start (s)
    end: float              # predicted speech end (s)
    labels: np.ndarray      # per-frame decision: 1 = speech, 0 = non-speech
    energy: HistogramThreshold
    centroid: HistogramThreshold


class HistogramVAD:
    """
    Voice activity detector with adaptive thresholds taken from the histogram of two features:
    Short-Time Energy (STE) and Spectral Centroid (SC).

    For each feature T = (W * M1 + M2) / (W + 1), where M1 and M2 are the first and second local
    maxima of the smoothed histogram. A frame is speech when STE >= T_E AND SC >= T_C.
    Post-processing: silences shorter than MIN_SILENCE_MS are filled, then both ends are padded.
    """

    def __init__(self, bins=100, smoothing_width=5, weight_e=5.0, weight_c=5.0,
                 pad_start=PAD_START, pad_end=PAD_END):
        self.bins = bins
        self.smoothing_width = smoothing_width
        self.weight_e = weight_e
        self.weight_c = weight_c
        self.pad_start = pad_start
        self.pad_end = pad_end

    def _smooth(self, counts):
        """Moving average over smoothing_width bins (the window is truncated at the edges)."""
        half = self.smoothing_width // 2
        return np.array([counts[max(0, i - half): i + half + 1].mean() for i in range(len(counts))])

    @staticmethod
    def _first_two_peaks(smooth):
        """
        Indices of the first two local maxima along the value axis (paper, Sec. II-C, step 4).
        Bin 0 may count as a peak, since the silence peak usually sits there.
        """
        peaks = []
        if smooth[0] >= smooth[1]:
            peaks.append(0)
        peaks += [i for i in range(1, len(smooth) - 1)
                  if smooth[i] > smooth[i - 1] and smooth[i] >= smooth[i + 1]]
        if len(peaks) >= 2:
            return peaks[0], peaks[1]

        # Fallback for flat/unimodal histograms: the highest bin plus the highest bin more than 2 bins away
        first = int(np.argmax(smooth))
        candidates = [i for i in range(len(smooth)) if abs(i - first) > 2]
        second = max(candidates, key=lambda i: smooth[i]) if candidates else min(first + 3, len(smooth) - 1)
        return tuple(sorted((first, second)))

    def estimate_threshold(self, values, weight):
        """Build the smoothed histogram of one feature and derive M1, M2 and the threshold T."""
        counts, edges = np.histogram(values, bins=self.bins, range=(float(values.min()), float(values.max())))
        centers = (edges[:-1] + edges[1:]) / 2.0
        smooth = self._smooth(counts)
        i1, i2 = self._first_two_peaks(smooth)
        m1, m2 = float(centers[i1]), float(centers[i2])
        return HistogramThreshold((weight * m1 + m2) / (weight + 1.0), m1, m2, centers, smooth)

    @staticmethod
    def _fill_short_silences(labels, first, last, min_frames):
        """In place: turn silent gaps shorter than min_frames, between first and last speech frame, into speech."""
        idx = first
        while idx <= last:
            if labels[idx] == 1:
                idx += 1
                continue
            gap_start = idx
            while idx <= last and labels[idx] == 0:
                idx += 1
            if idx - gap_start < min_frames:
                labels[gap_start:idx] = 1

    def predict(self, sample):
        """Detect the speech region of a Sample and return a VADResult."""
        energy = self.estimate_threshold(sample.ste, self.weight_e)
        centroid = self.estimate_threshold(sample.sc, self.weight_c)
        labels = ((sample.ste >= energy.value) & (sample.sc >= centroid.value)).astype(int)

        speech = np.flatnonzero(labels)
        if len(speech) == 0:
            return VADResult(0.0, 0.0, labels, energy, centroid)
        first, last = int(speech[0]), int(speech[-1])

        self._fill_short_silences(labels, first, last, int(round(MIN_SILENCE_MS / HOP_MS)))

        # Pad the two ends only; silences >= MIN_SILENCE_MS inside the region stay untouched
        start_idx = max(0, first - self.pad_start)
        end_idx = min(len(labels) - 1, last + self.pad_end)
        labels[start_idx:first] = 1
        labels[last + 1:end_idx + 1] = 1

        start = float(sample.frame_starts[start_idx])
        end = min(sample.duration, float(sample.frame_starts[end_idx]) + FRAME_MS / 1000.0)
        return VADResult(round(start, 4), round(end, 4), labels, energy, centroid)

In [ ]:
# Cell 4: Evaluation helpers and TRAIN results
@dataclass
class Evaluation:
    """Prediction for one file together with its boundary errors (predicted - ground truth, in ms)."""
    sample: Sample
    result: VADResult
    d_start: float
    d_end: float

    @property
    def mae(self):
        return (abs(self.d_start) + abs(self.d_end)) / 2.0

    @property
    def rmse(self):
        return float(np.sqrt((self.d_start ** 2 + self.d_end ** 2) / 2.0))


def evaluate(samples, vad):
    """Run the VAD on every sample and compute its boundary errors."""
    evaluations = []
    for sample in samples:
        result = vad.predict(sample)
        d_start = (result.start - sample.gt[0]) * 1000.0
        d_end = (result.end - sample.gt[1]) * 1000.0
        evaluations.append(Evaluation(sample, result, d_start, d_end))
    return evaluations


def mean_mae(evaluations):
    return float(np.mean([e.mae for e in evaluations]))


def mean_rmse(evaluations):
    return float(np.mean([e.rmse for e in evaluations]))


def print_report(title, evaluations):
    """Print a per-file table of thresholds, boundaries and errors, followed by the mean MAE / RMSE."""
    header = (f"{'File':<16} | {'T_E (STE)':<10} | {'T_C (SC)':<10} | {'Ground truth':<16} | {'Predicted':<16} | "
              f"{'ΔStart':<9} | {'ΔEnd':<9} | {'MAE (ms)':<9} | {'RMSE (ms)':<9}")
    width = len(header)
    print("=" * width)
    print(f"{title:^{width}}")
    print("=" * width)
    print(header)
    print("-" * width)
    for e in evaluations:
        gt = f"[{e.sample.gt[0]:.2f}, {e.sample.gt[1]:.2f}]"
        pred = f"[{e.result.start:.2f}, {e.result.end:.2f}]"
        print(f"{e.sample.name:<16} | {e.result.energy.value:<10.4f} | {e.result.centroid.value:<10.4f} | "
              f"{gt:<16} | {pred:<16} | {e.d_start:+7.1f}ms | {e.d_end:+7.1f}ms | {e.mae:9.2f} | {e.rmse:9.2f}")
    print("-" * width)
    print(f"Mean MAE = {mean_mae(evaluations):.2f} ms  |  Mean RMSE = {mean_rmse(evaluations):.2f} ms")
    print("=" * width)


vad = HistogramVAD()
train_eval = evaluate(train_set, vad)
print_report("TRAIN SET EVALUATION", train_eval)

In [ ]:
# Cell 5: TEST results (report only: the configuration is never selected using these numbers)
test_eval = evaluate(test_set, vad)
print_report("TEST SET EVALUATION", test_eval)

In [ ]:
# Cell 6: Plot TEST results, one figure per file:
#   (1) waveform + STE + threshold T_E      (2) spectral centroid + threshold T_C
#   (3) histograms of both features (how M1, M2 and T are found)
C_WAVE, C_STE, C_SC = "#5b6b7a", "#e8871e", "#1f9e89"
C_GT, C_PRED, C_SPEECH = "#d62728", "#1f5fbf", "#2ca02c"


def draw_boundaries(ax, gt, pred):
    for t in gt:
        ax.axvline(t, color=C_GT, linestyle="--", linewidth=1.6)
    for t in pred:
        ax.axvline(t, color=C_PRED, linestyle="-", linewidth=1.6, alpha=0.9)


def label_threshold(ax, x, y, text, color):
    ax.text(x, y, text, color=color, va="bottom", ha="right", fontsize=8.5,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.0))


def draw_histogram(ax, thr, color, title, xlabel, log_scale=False):
    y = np.maximum(thr.smooth, 0.1) if log_scale else thr.smooth
    base = 0.1 if log_scale else 0
    ax.plot(thr.centers, y, color=color, linewidth=1.6)
    ax.fill_between(thr.centers, base, y, where=thr.centers >= thr.value, color=color, alpha=0.18)
    ax.axvline(thr.m1, color="#7f7f7f", linestyle=":", linewidth=1.6, label=f"M1 = {thr.m1:.3f}")
    ax.axvline(thr.m2, color="#9467bd", linestyle=":", linewidth=1.6, label=f"M2 = {thr.m2:.3f}")
    ax.axvline(thr.value, color="black", linestyle="--", linewidth=1.5, label=f"T = {thr.value:.3f}")
    if log_scale:
        ax.set_yscale("log")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frame count (smoothed)" + (", log scale" if log_scale else ""))
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)


def plot_evaluation(ev):
    sample, result = ev.sample, ev.result
    pred = (result.start, result.end)
    t_wave = np.arange(len(sample.signal)) / sample.fs
    t_frame = sample.frame_starts + FRAME_MS / 2000.0
    is_speech = result.labels == 1
    x_max = max(t_wave[-1], t_frame[-1])

    fig = plt.figure(figsize=(14, 10))
    left, right = 0.105, 0.935
    gs_top = fig.add_gridspec(2, 1, left=left, right=right, top=0.835, bottom=0.38, hspace=0.12,
                              height_ratios=[3.0, 1.8])
    gs_bot = fig.add_gridspec(1, 2, left=left, right=right, top=0.27, bottom=0.07, wspace=0.28)

    # (1) Waveform + STE
    ax1 = fig.add_subplot(gs_top[0])
    ax1.plot(t_wave, sample.signal, color=C_WAVE, linewidth=0.5)
    ax1.fill_between(t_frame, -1.05, 1.05, where=is_speech, step="mid", color=C_SPEECH, alpha=0.12)
    draw_boundaries(ax1, sample.gt, pred)
    ax1.set_ylim(-1.05, 1.05)
    ax1.set_ylabel("Amplitude")
    ax1.grid(alpha=0.25)
    ax1b = ax1.twinx()
    ax1b.plot(t_frame, sample.ste, color=C_STE, linewidth=1.3)
    ax1b.axhline(result.energy.value, color=C_STE, linestyle=":", linewidth=1.6)
    ax1b.set_ylim(0, 1.05)
    ax1b.set_ylabel("STE / max(STE)", color=C_STE)
    ax1b.tick_params(axis="y", colors=C_STE)
    label_threshold(ax1b, x_max * 0.995, result.energy.value, f"T_E = {result.energy.value:.3f}", C_STE)

    # (2) Spectral centroid
    ax2 = fig.add_subplot(gs_top[1], sharex=ax1)
    ax2.plot(t_frame, sample.sc, color=C_SC, linewidth=1.3)
    ax2.axhline(result.centroid.value, color=C_SC, linestyle=":", linewidth=1.6)
    ax2.fill_between(t_frame, 0, 1.05, where=is_speech, step="mid", color=C_SPEECH, alpha=0.12)
    draw_boundaries(ax2, sample.gt, pred)
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("SC / max(SC)", color=C_SC)
    ax2.tick_params(axis="y", colors=C_SC)
    label_threshold(ax2, x_max * 0.995, result.centroid.value, f"T_C = {result.centroid.value:.3f}", C_SC)
    ax2.set_xlabel("Time (s)")
    ax2.set_xlim(0, x_max)
    ax2.grid(alpha=0.25)
    plt.setp(ax1.get_xticklabels(), visible=False)

    # (3) Histograms of both features
    draw_histogram(fig.add_subplot(gs_bot[0]), result.energy, C_STE,
                   "STE histogram: finding M1, M2 and threshold T_E", "STE / max(STE)", log_scale=True)
    draw_histogram(fig.add_subplot(gs_bot[1]), result.centroid, C_SC,
                   "Spectral Centroid histogram: finding M1, M2 and threshold T_C", "SC / max(SC)")

    # Title and shared legend
    fig.suptitle(sample.name, fontsize=14, fontweight="bold", y=0.985)
    fig.text(0.5, 0.945,
             f"MAE = {ev.mae:.1f} ms  ·  RMSE = {ev.rmse:.1f} ms  ·  ΔStart = {ev.d_start:+.0f} ms  ·  ΔEnd = {ev.d_end:+.0f} ms"
             f"      |      Frame {FRAME_MS} ms  ·  Hop {HOP_MS} ms  ·  Min silence ≥ {MIN_SILENCE_MS} ms",
             ha="center", fontsize=9.5, color="0.25")
    handles = [
        Line2D([0], [0], color=C_WAVE, linewidth=1.2, label="Waveform"),
        Line2D([0], [0], color=C_STE, linewidth=1.6, label="Normalized STE"),
        Line2D([0], [0], color=C_SC, linewidth=1.6, label="Normalized SC"),
        Line2D([0], [0], color="0.3", linestyle=":", linewidth=1.6, label="Adaptive threshold"),
        Line2D([0], [0], color=C_GT, linestyle="--", linewidth=1.6, label="Ground truth (LAB)"),
        Line2D([0], [0], color=C_PRED, linewidth=1.6, label="Predicted boundary"),
        Patch(facecolor=C_SPEECH, alpha=0.3, label="Predicted speech region"),
    ]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.925), ncol=7, fontsize=8.5, frameon=False)

    fig_path = OUTPUT_DIR / f"{Path(sample.name).stem}.png"
    fig.savefig(fig_path, dpi=150)
    print(f"Saved figure: {fig_path}")
    plt.show()


for ev in test_eval:
    plot_evaluation(ev)

In [ ]:
# Cell 7: Boundary-padding sweep (0-4 frames per side).
# PAD_START / PAD_END are not given by the paper, so this sweep is the evidence for the values in Cell 1.
# Uses TRAIN only: TEST is deliberately not evaluated here, so it cannot influence the choice.
MAX_PAD = 4

sweep = []
for pad_start in range(MAX_PAD + 1):
    for pad_end in range(MAX_PAD + 1):
        candidate = HistogramVAD(pad_start=pad_start, pad_end=pad_end)
        sweep.append((mean_mae(evaluate(train_set, candidate)), pad_start, pad_end))
sweep.sort()

print(f"{'pad_start':<10} {'pad_end':<8} {'Train MAE (ms)':>14}")
print("-" * 34)
for train_mae, pad_start, pad_end in sweep[:8]:
    print(f"{pad_start:<10} {pad_end:<8} {train_mae:>14.2f}")
print("\nTop configurations often tie (1 frame = 10 ms); on a tie, pick the simplest (symmetric, smallest).")